In [5]:
CSV_PATH  = "jobs_final.csv"              # path to your jobs CSV file
ES_URL    = "https://localhost:9200" # Elasticsearch URL
ES_USER   = "elastic"
ES_PASS   = "4FPqLpFFwvF-5jfcyUlm"
SLEEP_SEC = 0.0                     # optional delay between rows (0 = max speed)

## 1 — Imports

In [6]:
import time
import json
import hashlib

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch

## 2 — Load model

In [7]:
print("Loading JobBERT — this may take a minute on first run...")
jobbert = SentenceTransformer("TechWolf/JobBERT-v2")
DIM = jobbert.get_embedding_dimension()  
EMBEDDING_DIM = DIM * 3                           
print(f"Model ready. Segment dim: {DIM} | Final embedding dim: {EMBEDDING_DIM}")

Loading JobBERT — this may take a minute on first run...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model ready. Segment dim: 1024 | Final embedding dim: 3072


## 3 — Connect to Elasticsearch & create index

In [ ]:
es = Elasticsearch(
    ES_URL,
    basic_auth=(ES_USER, ES_PASS),
    verify_certs=False,
    ssl_show_warn=False,
)

In [ ]:
INDEX_NAME = "jobs_strat2_normal"

mapping = {
    "mappings": {
        "properties": {
            "job_id":          {"type": "keyword"},
            "job_title":       {"type": "text"},
            "normal_job_title":       {"type": "text"},
            "job_description": {"type": "text"},
            "skills":          {"type": "keyword"},
            "embedding": {
                "type":       "dense_vector",
                "dims":       EMBEDDING_DIM,
                "index":      True,
                "similarity": "cosine",
            },
        }
    }
}

if not es.indices.exists(index=INDEX_NAME):
    es.indices.create(index=INDEX_NAME, body=mapping)
    print(f"Created index '{INDEX_NAME}'")
else:
    print(f"Index '{INDEX_NAME}' already exists — skipping creation")

Index 'jobs' already exists — skipping creation


## 4 — Load & preview CSV

In [6]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df):,} rows")
df.head(3)

Loaded 39,650 rows


,Unnamed: 0,job_link,job_title,company,job_location,city,country,job_position,job_level,job_type,job_skills,job_summary,is_tech
0,52,https://www.linkedin.com/jobs/view/sr-hardware...,"Sr. Hardware Simulation Platform Engineer, Cha...",Tesla,"Palo Alto, CA",Alameda,United States,Processor,Mid senior,Onsite,NaN,NaN,True
1,89,https://www.linkedin.com/jobs/view/senior-soft...,Senior Software Engineer - Backend,Walmart,"Reston, VA",Bethesda-Chevy Chase,United States,Chief Computer Programmer,Mid senior,Onsite,NaN,NaN,True
2,106,https://www.linkedin.com/jobs/view/senior-netw...,Senior Network Engineering Lead - Remote,"Ryder System, Inc.","Richmond, VA",Montpelier,United States,Frame Hand,Mid senior,Onsite,"Network Engineering, Cloud Computing, Azure, A...",Job Seekers can review the Job Applicant Priva...,True


In [7]:
# Null counts for the columns we'll use
df[["job_link", "job_title", "job_skills", "job_summary"]].isnull().sum().rename("nulls")

job_link          0
job_title         0
job_skills     1916
job_summary    1876
Name: nulls, dtype: int64

## 5 — Clean & prepare

In [ ]:
def make_job_id(row) -> str:
    """Use job_link as ID; fall back to an MD5 hash of title+company+location."""
    link = str(row.get("job_link", "")).strip()
    if link and link != "nan":
        return link
    seed = f"{row.get('job_title','')}|{row.get('company','')}|{row.get('job_location','')}" 
    return "job_" + hashlib.md5(seed.encode()).hexdigest()[:12]


def generate_job_embedding(title: str, description: str, skills: list[str]) -> list[float]:
    """
    Concatenate JobBERT encodings of title, description, and skills
    into a single 3072-dim vector: [title(1024) | desc(1024) | skills(1024)]
    """
    title_emb  = jobbert.encode(title       if title       else "unknown")
    desc_emb   = jobbert.encode(description if description else "")
    skills_emb = jobbert.encode(" ".join(skills) if skills else "general")
    return np.concatenate([title_emb, desc_emb, skills_emb]).tolist()


# Apply cleaning
df["_job_id"]  = df.apply(make_job_id, axis=1)
df["_title"]   = df["normal_job_title"].fillna("Other").str.strip()
df["_skills"]  = df["job_skills"].apply(lambda x: [s.strip() for s in x.split("|") if s.strip()])
df["_description"] = df["job_description"].fillna("").str.strip()

before = len(df)
df = df[df["_title"] != ""].reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with no title. Ready to embed: {len(df):,}")

Dropped 0 rows with no title. Ready to embed: 39,650


## 6 — Dry run (first 3 rows)

In [ ]:
for _, row in df.head(3).iterrows():
    emb = generate_job_embedding(row["_title"], row["_description"], row["_skills"])
    print(f"[OK] '{row['_title']}' → {len(emb)} dims | skills: {row['_skills'][:3]}")

[OK] 'Sr. Hardware Simulation Platform Engineer, Chassis Controls & Powertrain' → 3072 dims | skills: []
[OK] 'Senior Software Engineer - Backend' → 3072 dims | skills: []
[OK] 'Senior Network Engineering Lead - Remote' → 3072 dims | skills: ['Network Engineering', 'Cloud Computing', 'Azure']


## 7 — Bulk embed & index
Embeds and indexes every row directly into Elasticsearch. Errors are collected without stopping the run.

In [10]:
indexed = 0
errors  = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Embedding & indexing"):
    job_id = row["_job_id"]
    try:
        embedding = generate_job_embedding(
            row["_title"],
            row["_summary"],
            row["_skills"],
        )

        document = {
            "job_id":          job_id,
            "job_title":       row["_title"],
            "job_description": row["_summary"],
            "skills":          row["_skills"],
            "embedding":       embedding,
        }

        es.index(index=INDEX_NAME, id=job_id, document=document)
        indexed += 1

    except Exception as exc:
        errors.append({"job_id": job_id, "title": row["_title"], "error": str(exc)})

    if SLEEP_SEC:
        time.sleep(SLEEP_SEC)

print(f"\nDone.  Indexed: {indexed:,} | Errors: {len(errors)}")

Embedding & indexing:   0%|          | 0/39650 [00:00<?, ?it/s]


Done.  Indexed: 39,650 | Errors: 0


## 8 — Inspect errors

In [11]:
if errors:
    err_df = pd.DataFrame(errors)
    display(err_df)
    err_df.to_csv("embedding_errors.csv", index=False)
    print("Saved to embedding_errors.csv")
else:
    print("No errors — all jobs indexed successfully.")

No errors — all jobs indexed successfully.


## 9 — Verify: kNN search test

In [16]:
# Build a query embedding from the first row's title + skills
sample = df.iloc[0]

query_vector = np.concatenate([
    jobbert.encode(sample["_title"]),
    np.zeros(DIM),                                                         # zero-pad description slot
    jobbert.encode(" ".join(sample["_skills"]) if sample["_skills"] else "general"),
]).tolist()

response = es.search(
    index=INDEX_NAME,
    body={
        "knn": {
            "field":          "embedding",
            "query_vector":   query_vector,
            "k":              5,
            "num_candidates": 50,
        },
        "_source": ["job_id", "job_title", "skills"],
    },
)

print(f"Top 5 matches for '{sample['_title']}':")
for hit in response["hits"]["hits"]:
    src = hit["_source"]
    print(f"  [{hit['_score']:.4f}] {src['job_title']}")

Top 5 matches for 'Sr. Hardware Simulation Platform Engineer, Chassis Controls & Powertrain':
  [0.9406] Sr. Hardware Simulation Platform Engineer, Chassis Controls & Powertrain
  [0.8505] Sr. Mechanical Engineer, Vehicle & Firmware, Hardware Test Engineering
  [0.8293] Senior Software Engineer, Embedded Systems/Firmware, Platforms Infrastructure Engineering
  [0.8226] Senior System Software Engineer - Automotive AI Applications
  [0.8222] Sr. Flight Control Systems Engineer


## 10 — Index stats

In [18]:

count = es.count(index=INDEX_NAME)["count"]
print(f"Total documents in '{INDEX_NAME}': {count}")

Total documents in 'jobs': 39650
